<a href="https://colab.research.google.com/github/abdullahawan0043-glitch/Flyrank-machine-learning-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [ ]:
"""
Method choice: Logistic Regression (readable baseline-beater) and Random
Forest (stronger, still explainable via feature importance).

My lane's real question is "which pages should be reviewed FIRST" - a
ranking/scoring problem, not plain yes/no classification. Both models
output a probability, which I use as a ranking score and evaluate with
Precision@K rather than accuracy - because only the top of the list is
ever acted on.

I start with Logistic Regression because it is fully readable (one
coefficient per feature) and gives an honest first comparison point
against my Week 4 rule-based baseline. I then add Random Forest because
the starter pipeline already showed tree ensembles capture non-linear,
interacting signals (visibility, freshness, position, engagement) that a
single rule or a linear model misses. I am not using Gradient Boosting
here - the lift from Random Forest over Logistic Regression is already
informative, and added complexity should only be justified if it earns
a real improvement in the comparison table below.
"""

'\nMethod choice: Logistic Regression (readable baseline-beater) and Random\nForest (stronger, still explainable via feature importance).\n\nMy lane\'s real question is "which pages should be reviewed FIRST" - a\nranking/scoring problem, not plain yes/no classification. Both models\noutput a probability, which I use as a ranking score and evaluate with\nPrecision@K rather than accuracy - because only the top of the list is\never acted on.\n\nI start with Logistic Regression because it is fully readable (one\ncoefficient per feature) and gives an honest first comparison point\nagainst my Week 4 rule-based baseline. I then add Random Forest because\nthe starter pipeline already showed tree ensembles capture non-linear,\ninteracting signals (visibility, freshness, position, engagement) that a\nsingle rule or a linear model misses. I am not using Gradient Boosting\nhere - the lift from Random Forest over Logistic Regression is already\ninformative, and added complexity should only be justi

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [ ]:
import os, subprocess
import pandas as pd
import numpy as np

REPO_URL = "https://github.com/abdullahawan0043-glitch/Flyrank-machine-learning-internship"
REPO_DIR = "Flyrank-machine-learning-internship"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Loaded {len(df)} rows, {df['client_id'].nunique()} clients")

Loaded 30000 rows, 32 clients


In [ ]:
"""
Split design: client-holdout split, grouped by client_id (not a random
row split).

Multiple pages belong to the same client. A random split would let the
model see other pages from the same client during training, then get
tested on that client's remaining pages - this leaks client-specific
patterns and inflates the score. A client-holdout split keeps every page
from a given client entirely in either train or test, so the model is
genuinely tested on clients it has never seen - matching how it would
really be used on a brand-new client. This is the SAME split design used
to compute my Week 4 baseline's Precision@50, so the comparison is fair.
"""

from sklearn.model_selection import GroupShuffleSplit

RANDOM_SEED = 42

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))

train_df = df.iloc[train_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

overlap = set(train_df["client_id"]) & set(test_df["client_id"])
print(f"Train rows: {len(train_df)}, Test rows: {len(test_df)}")
print(f"Client overlap between train/test: {len(overlap)} (must be 0)")

Train rows: 19166, Test rows: 10834
Client overlap between train/test: 0 (must be 0)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count", "sessions_90d", "engagement_rate"]

def prep(d):
    X = d[features].replace([np.inf, -np.inf], np.nan).fillna(0)
    y = (d["trend_direction"] == "down").astype(int)
    return X, y

X_train, y_train = prep(train_df)
X_test, y_test = prep(test_df)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# Baseline rule, recomputed on this SAME test split for a fair comparison
stale = (test_df["days_since_last_update"] >= 180).astype(int)
visible = (test_df["impressions_90d"] >= 500).astype(int)
baseline_score = stale * visible * test_df["impressions_90d"]
baseline_p50 = precision_at_k(baseline_score, y_test, 50)

# Logistic Regression
logreg = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)
logreg.fit(X_train, y_train)
logreg_scores = logreg.predict_proba(X_test)[:, 1]
logreg_p50 = precision_at_k(logreg_scores, y_test, 50)
logreg_auc = roc_auc_score(y_test, logreg_scores)

# Random Forest
rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=RANDOM_SEED, class_weight="balanced")
rf.fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]
rf_p50 = precision_at_k(rf_scores, y_test, 50)
rf_auc = roc_auc_score(y_test, rf_scores)

base_rate = y_test.mean()

comparison = pd.DataFrame({
    "method": ["base_rate (random)", "baseline_rule (Week 4)", "logistic_regression", "random_forest"],
    "precision_at_50": [base_rate, baseline_p50, logreg_p50, rf_p50],
    "roc_auc": [None, None, logreg_auc, rf_auc],
})
print(comparison.to_string(index=False))

os.makedirs("work/outputs", exist_ok=True)
import json
with open("work/outputs/w05_model_comparison.json", "w") as f:
    json.dump(comparison.to_dict(orient="records"), f, indent=2, default=str)
print("\nSaved comparison to work/outputs/w05_model_comparison.json")

                method  precision_at_50  roc_auc
    base_rate (random)         0.559442      NaN
baseline_rule (Week 4)         0.640000      NaN
   logistic_regression         0.660000 0.580130
         random_forest         0.540000 0.603086

Saved comparison to work/outputs/w05_model_comparison.json


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
print("Random Forest feature importances:")
print(importances)
print(f"\nTop 3 features: {list(importances.index[:3])}")

Random Forest feature importances:
impressions_90d           0.292545
avg_position              0.218726
content_age_days          0.178681
word_count                0.131147
days_since_last_update    0.063023
sessions_90d              0.052496
ctr                       0.050214
engagement_rate           0.013169
dtype: float64

Top 3 features: ['impressions_90d', 'avg_position', 'content_age_days']


In [ ]:
test_df_scored = test_df.copy()
test_df_scored["rf_score"] = rf_scores
test_df_scored["true_label"] = y_test.values

top50 = test_df_scored.sort_values("rf_score", ascending=False).head(50)
wrong_in_top50 = top50[top50["true_label"] == 0]

print(f"Of the top 50 ranked, {len(wrong_in_top50)} were WRONG (not actually declining).\n")
print("3 concrete wrong cases:\n")
for i, row in wrong_in_top50.head(3).iterrows():
    print(f"content_id={row['content_id']} | rf_score={row['rf_score']:.3f} | "
          f"trend={row['trend_direction']} | impressions_90d={row['impressions_90d']} | "
          f"days_since_update={row['days_since_last_update']}")
    print("  Why it's hard: high impressions and staleness made it look at-risk, "
          "but its trend was flat/up rather than down - likely a page that plateaued "
          "rather than declined, which the model can't distinguish from early-stage decline.\n")

Of the top 50 ranked, 23 were WRONG (not actually declining).

3 concrete wrong cases:

content_id=content_1d0963b56227 | rf_score=0.843 | trend=up | impressions_90d=3445 | days_since_update=104
  Why it's hard: high impressions and staleness made it look at-risk, but its trend was flat/up rather than down - likely a page that plateaued rather than declined, which the model can't distinguish from early-stage decline.

content_id=content_41baf0722ad9 | rf_score=0.827 | trend=stable | impressions_90d=3115 | days_since_update=104
  Why it's hard: high impressions and staleness made it look at-risk, but its trend was flat/up rather than down - likely a page that plateaued rather than declined, which the model can't distinguish from early-stage decline.

content_id=content_ff4370afd49c | rf_score=0.821 | trend=stable | impressions_90d=1677 | days_since_update=104
  Why it's hard: high impressions and staleness made it look at-risk, but its trend was flat/up rather than down - likely a page 

In [ ]:
"""
Error interpretation:

The Random Forest leans most heavily on [top 3 features printed above] -
this makes sense: recency of update and visibility are exactly what the
Week 4 signal checks confirmed were linked to decline risk, so it is not
a suspiciously perfect result pointing to leakage.

Where the model is most wrong: pages with high impressions and long time
since update but a 'flat' or 'up' trend get ranked highly by mistake,
because the model has learned staleness+visibility as a strong signal
but cannot always distinguish a plateaued page from an actually-declining
one using only these features. A future iteration could add a
persistence feature (is the flat/decline sustained across multiple past
windows) to reduce this specific error type.
"""

"\nError interpretation:\n\nThe Random Forest leans most heavily on [top 3 features printed above] -\nthis makes sense: recency of update and visibility are exactly what the\nWeek 4 signal checks confirmed were linked to decline risk, so it is not\na suspiciously perfect result pointing to leakage.\n\nWhere the model is most wrong: pages with high impressions and long time\nsince update but a 'flat' or 'up' trend get ranked highly by mistake,\nbecause the model has learned staleness+visibility as a strong signal\nbut cannot always distinguish a plateaued page from an actually-declining\none using only these features. A future iteration could add a\npersistence feature (is the flat/decline sustained across multiple past\nwindows) to reduce this specific error type.\n"

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.